In [0]:
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("merge_keys","")
dbutils.widgets.text("load_type","")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

#print(landing_timestamp)
#print(table_name)
#print(merge_keys)
#print(load_type)

In [0]:
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp='{landing_timestamp}'")

In [0]:
from pyspark.sql.functions import current_timestamp

target_table = f"adb_caresync.silver.{table_name}"
normalized_load_type = (load_type or 'FULL').strip().upper()
merge_key_list =[key.strip() for key in merge_keys.split(",") if key.strip()]
print(merge_key_list)

if normalized_load_type == 'MERGE':
    if not merge_key_list:
        raise ValueError("Merger key must be provide when load type is 'MERGE'")

    missing_merge_key = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_key:
        raise ValueError(f"merge_key not found in source data :{missing_merge_key}")

updated_df=bronze_df.withColumn("insert_timestamp",current_timestamp())\
                   .withColumn("updated_timestamp",current_timestamp())
    

In [0]:
if normalized_load_type == 'FULL':
   updated_df.write.mode("overwrite").saveAsTable(target_table)
elif normalized_load_type == 'APPEND':   
   updated_df.write.mode("append").saveAsTable(target_table) 
elif normalized_load_type == 'MERGE':  
    if not spark.catalog.tableExists(target_table):
       updated_df.write.mode("overwrite").saveAsTable(target_table)   
    else:
        source_view ="silver_merge_source"    
        updated_df.createOrReplaceTempView(source_view)

        merge_condition = " AND " .join(
                          [f"target.`{key}` = source.`{key}` " for key in merge_key_list]
                   )
        #print(merge_condition)
        update_assignment = ",".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timstamp"
            ]
            )
       # print(update_assignment)
        insert_columns = ", ".join( [f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join( [f"source.`{column}`" for column in updated_df.columns])
       # print(insert_columns)
       # print(insert_values)

        query = f"""
                   MERGE INTO {target_table} AS target 
                   USING {source_view} AS source
                   ON {merge_condition}
                   WHEN MATCHED THEN UPDATE SET {update_assignment}
                   WHEN NOT MATCHED THEN INSERT({insert_columns})
                                         VALUES({insert_values})
                  """
       #print(query) 
        spark.sql(query)

else:
    raise ValueError("Load_type must be one of : FULL ,APPEND,MERGE")        

In [0]:
record_count =updated_df.count()
print(record_count)
dbutils.notebook.exit(str(record_count))